# This Agent will judge your bias! - Pycon Portugal 2026

# Setup Check

Before starting, make sure your `.env` file is in the same folder as this notebook.

The `.env` file should contain your Hugging Face token:

```text
HF_TOKEN=your_token_here
```

Run these 2 cells below first to verify that your environment is ready:
- Hugging Face token is loaded
- Required packages are available
- The notebook is ready to run

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")


print(r"""
   /\_/\\
  ( •ᴗ• )
  / >💜
HF token loaded
""")

In [ ]:
import langchain
import langgraph

print("⚧️  Ready to challenge AI bias!")

## ⚠️ Setup Troubleshooting

If you see an error while running the setup cells:

1. Check that your `.env` file is in the same folder as this notebook.
2. Make sure the file contains your Hugging Face token, and you saved it:

```text
HF_TOKEN=your_token_here
```

# Intro to Agents 

#### LLMs vs Agents

**LLM** - Large Language Model - predicts text, generates human-style "natural" language. Trained on vast amounts of data. "Knowledge" limited to training data, cannot reason and can hallucinate

Examples: Claude Sonnet, GPT-, Qwen3, etc

##### Connecting to an LLM with LangChain:

In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

def connect_to_llm():
    try:
        llm = HuggingFaceEndpoint(
            repo_id="meta-llama/Llama-3.1-8B-Instruct", #see what happens if you change this model 
            huggingfacehub_api_token=HF_TOKEN,
            provider="auto",
            task="text-generation",
            temperature=0.7, #control randomness (0= same answer every time, 0.7= more creative)
            max_new_tokens=500) #change this to be less to save money, more for more verbose output
        print("Successfully connected to HuggingFace LLM :))")
        return ChatHuggingFace(llm=llm)
        
    except Exception as e:
        print(f"Error connecting to LLM: {e}")
        raise

In [ ]:
llm = connect_to_llm()

#### Invoke vs. stream:

LangChain has two main ways of accessing the response of the model to a prompt, `invoke` and `stream`. `invoke` returns the entire output at once, and is the method we will be working with today:

In [ ]:
response = llm.invoke('Hello')
response

You can individually access the attibutes in the response by using e.g.:

`llm.invoke('Hello').content` or `llm.invoke('Hello').response_metadata` etc. 

To see the output in a nice human-readable way, you can also use the inbuilt `.pretty_print()` method:

In [ ]:
response.pretty_print()

#### Messages

The response above was generated for us from just passing a string to our LLM interface. It mentions that it is an AI Message, and if we check the data type, this is also the type of object which we are dealing with:

In [ ]:
type(response)

This is a special kind of structured data object which contains not only the text of the message, but also the **role** and some **metadata** that can let the LLM (and us!) know that it is a product of the model. 

Langchain has some other Message types, the most common being:
- `HumanMessage` - lets the model know we the human are talking to it
- `SystemMessage` - system-level prompt to specify instructions for response behaviour
- `AnyMessage` - placeholder class that allows us to hold message data and assign an annotation of the role
- `ToolMessage` - explained below



We can also pass in our prompt as one of these objects, or even a list of multiple messages. Note that if we are using `Message` objects from LangChain we must pass in a list, even if we are only passing in one message:

In [ ]:
from langchain_core.messages import HumanMessage
human_prompt = HumanMessage(content="Hello, how are you today?")
llm.invoke([human_prompt]).pretty_print()

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
system_prompt = SystemMessage(content='you are a passionate poet who writes in Portuguese')
human_prompt = HumanMessage(content="Hi, please write me a limerick about python")

llm.invoke([system_prompt, human_prompt]).pretty_print()

For more details, see the `Message` documentation from LangChain [here](https://docs.langchain.com/oss/python/langchain/messages)

#### Memory

An LLM in and of itself **has no memory**:

In [ ]:
human_intro = HumanMessage(content="Hello, my name is Anastasia")
llm.invoke([human_intro]).pretty_print()

print('\n\n---- next message exchange ----\n\n')


llm.invoke('Hello, what is my name?').pretty_print()

A basic way that we can keep a memory of all the previous messages that were exchanged is to keep appending them to a message list that we pass to the LLM at each subsequent call:


In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# Define the assistant's role and behavior
system_prompt = SystemMessage(content="You are a friendly usher at the PyCon 2026 conference, and you answer in Portuguese only.")

# Start the conversation history with the system instructions
messages_sequence = [system_prompt]

# Add previous user and assistant messages to preserve context
messages_sequence.append(human_intro)
messages_sequence.append(AIMessage(content="Hello Anastasia, nice to meet you! How can I assist you today?"))

# Add the new user question
messages_sequence.append(HumanMessage(content="Hello, what is my name?"))

llm.invoke(messages_sequence).pretty_print()


### Agents

So far we have only been talking to a basic LLM. It is able to generate us a response based on some prompt(s), but it is not quite an agent. 

An Agent is an application that has (usually) an LLM working as its "brains" under the hood, but has additional functionality, such as *reasoning*, *decision-making* and *access to tools*. It is able to translate a human prompt into a plan for a sequence of steps and implement these to return a response. It can have power to take actions on the human user's behalf, and extend the functionality of the "vanilla" model under the hood.

In the next sections we will look into how to build agents of ever greater complexity:



## Intro to Langgraph

LangGraph is an open-source framework for building agentic applications using graphs to manage the flow of information. It gives us a lot of control over the structure and design of our agent. 

#### State

The main data structure in our LangGraph application is called the `State`. It is the **current status** of the data in our system, and **all the parts of our system can access it and change it**. As the conversation and workflow flows, the state is updated and acts as a persistent "shared memory" across the application.

We decide what structure we want our data to have in advance and create the State to be able to contain things like the user’s message, the model’s response, intermediate results, and any flags or metadata that the workflow needs to remember.

Typically, the State is defined as a `TypedDict` or a Pydantic model, which gives the workflow a clear structure for storing and updating information. Python does not enforce types, however for LLMs interacting with each other, this provides structure for the kinds of data that are being passed through the system.

#### Nodes and Edges

In LangGraph, a **node** is one step or function in the workflow. Each node performs a **specific job**, such as receiving input, calling an LLM, using a tool, making a decision, or formatting a response. **Nodes can read and update the graph's state**



An **edge** is the connection between nodes. It tells the workflow what should happen next. For example, after one node finishes, the edge can send the result to another node, or decide whether to loop back or stop.

A simple way to think about it is:
- **Nodes** = the individual tasks
- **Edges** = the paths that connect those tasks

Together, nodes and edges form the graph that defines how the agent moves through its workflow.

Two related special concepts are **tool nodes** and **conditional edges**. 
- A **tool node** is a node whose job is to call an external tool, such as a web search or a database query. 
- A **conditional edge** is an edge that sends the workflow to different next nodes **depending on a condition**, such as whether an answer was found or whether the process should continue.

## Building a Baby Agent

Now that we have the main terminology, it is time to build the first agent with langgraph! 

We can have a basic workflow that writes a message word by word:

We define the data schema for our agent, the `AgentState` to be a dictionary which contains a single message as a string.
(We can add more fields to the AgentState as needed, but for this example, we will keep it simple.)

In [ ]:
from typing import TypedDict

class AgentState(TypedDict):
    message: str

Next, we define two steps in our workflow. These are just python functions that will act on the AgentState. 
Each step takes the current state as input and returns the updated state as output.
The first step sets the message to "hello", and the second step appends " world" to the message.


In [ ]:
def first_step(state: AgentState) -> AgentState:
    print('First step is being executed, message now says "hello"')
    return {"message": "hello"}


def second_step(state: AgentState) -> AgentState:
    print('Second step is being executed, adding " world" to the message')
    return {"message": state["message"] + " world"}

And now we can combine the building blocks above into a single workflow, called a `StateGraph`
In Langgraph, it is important that the graph has a `START` and an `END`, which tell it when to begin and complete the workflow.

If we want to add a step as a node, we use `add_node` and pass in the name of our step as a string, and the function which is being run at that step.

To connect two nodes with an edge, we use `add_edge` and pass in the two nodes that are being connected, in order of the flow. Finally we `compile` the graph:

In [ ]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(AgentState)
# first tell langgraph which nodes we want to have in our graph:
workflow.add_node("first_step", first_step) #name of the node, and the function that will be executed when we reach this node
workflow.add_node("second_step", second_step) 

# and now we specify how to connect them:
workflow.add_edge(START, "first_step") #from start to first step
workflow.add_edge("first_step", "second_step") #from first step to second step
workflow.add_edge("second_step", END) #from second step to end

# finally put it all together:
graph = workflow.compile()

#### Visualising the Graph 

Langgraph lets us view the graph as an image. In jupyter you can just refer to the compiled graph, or you can use `.get_graph().draw_mermaid_png()`to display the image outside of jupyter, or to change the settings

In [ ]:
#For more flexibility, e.g. changing curve style:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

#can also just refer to the variable in jupyter:
#graph

Now we can see the workflow we designed, and now we have to run it. We can call `invoke()`, pass in an initial message, and then the functionalities of the nodes are executed in order:

In [ ]:
start_state = {"message": "hi"}
graph.invoke(start_state)

Notice:
- we have to pass in an **initial state** which conforms to the data structure the Graph is using. 
- we start out with `"hi"` and the first node of the graph *overwrites* this with `hello`, and the second node *appends* `world`. 
- **Each node of the graph can access the state and change it!**

## Giving it a choice

Sometimes the order of steps in the workflow is not set in advance, but might be determined *during* the invocation of the graph. In this case we can add **conditional edges** - these are choices in which path to take, which depend on the outcome of some process. 

So for example we can have a 50/50 probability of one or the other outcome:

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
import random

class AgentState(TypedDict):
    message: str


def write_hello(state: AgentState) -> AgentState:
    print('message now says "hello"')
    return {"message": "hello"}


def add_world(state: AgentState) -> AgentState:
    print('message now says "hello world!"')
    return {"message": state["message"] + " world!"}


def add_pycon_portugal(state: AgentState) -> AgentState:
    print('message now says "hello Pycon Portugal!"')
    return {"message": state["message"] + " Pycon Portugal!"}


def flip_a_coin(state: AgentState) -> str:
    print('tossing a coin to decide what to do next...')
    return "heads" if random.choice([True, False]) else "tails"

# As before we create our StateGraph:
workflow = StateGraph(AgentState)

#Now we have 3 nodes to add:
workflow.add_node("Write 'hello'", write_hello) #name of the node, and the function that will be executed when we reach this node
workflow.add_node("Add 'world'", add_world)
workflow.add_node("Add 'Pycon Portugal'", add_pycon_portugal)

# We start with node 1 as before:
workflow.add_edge(START, "Write 'hello'")

#And now we add the conditional edges
# These go from the first node to either add 'world' or add 'Pycon Portugal',
# based on the return value of our flip_a_coin function.

workflow.add_conditional_edges(
    "Write 'hello'", #from here
    flip_a_coin, #according to the result of this

    {
        "heads": "Add 'world'", #if heads, we go to the node with this name
        "tails": "Add 'Pycon Portugal'", # if tails, go to the node with this name
    },
)

#and no matter what happens, we want to end after either result:
workflow.add_edge("Add 'world'", END) 
workflow.add_edge("Add 'Pycon Portugal'", END)

app = workflow.compile()


The visualisation conveniently labels the conditional edges for us so we can see what is going on, also **definite** paths are shown with a **solid arrow** and **conditional** paths are shown with a **dashed** arrow:

In [ ]:
#Using this syntax we can change some of the visual aspects of the graph, e.g. the curve style:
from langchain_core.runnables.graph_mermaid import CurveStyle
display(Image(app.get_graph().draw_mermaid_png(curve_style=CurveStyle.BASIS)))

In [ ]:
app.invoke({'message': 'hi there'})

## Building an agent with an LLM

Now what we can do is set up a graph where our LLM is the "brain" node at the centre and has access to various options for decision-making.

The basic principles we saw above remain the same, but with the added power of us being able to use **human language** to interact with our program.

We can also have the nodes have more complex functionality, e.g. searching the web or a database, and can even have these processes running in parallel. We can add sub-graphs to our main graph which are themselves like a small "brain" ... the opportunities are vast!

Since the functionality of sending messages between humans and a central LLM "brain", or between sub-agents with different roles, and wanting to append the messages to a running conversation is so common, there is a pre-built State class from LangGraph called `MessagesState` which comes shipped with a lot of the functionality we need:
- it has a pre-built `messages` key, 
    - this is a list of `AnyMessage` objects 
    - which means all kinds of `Message` (System, Human, AI, Tool etc) can be added in any order
    - It uses the `add_messages` *reducer* by default, aka any messages generated anywhere in the graph get **appended** to the conversation (rather than **overwritten**), and each node can see the message history when it is triggered

## Giving it tools

We saw above that a node is more or less a python function.

There is a special type of python function that can mediate between human language, an LLM, and another piece of software which needs structured input, called a `tool`, and we can attach tools to our basic chat model to extend its functionality. 

We will see below how it is possible to combine all these concepts into a more complex system, and have the LLM at the heart of it have some judgement/decision-making capability:

#### Basic web search

We will start with a basic web search example. The "vanilla" LLM we are using was trained on a vast amount of data, but (depending on the model we are using) has access only to the information at the time of training. This means that if we ask it for current news or very up-to-date information, it will either hallucinate, or give us the answer that was correct at the time of training, or tell us that it doesn't know:

In [ ]:
llm.invoke('who is the uk prime minister at the moment?').pretty_print()

One thing we can do is to add an additional capability, or `tool` to our model, that it can **choose to use**, or **be forced to use** to give its response. 

Tools are useful because they allow the LLM to access **external information** and **perform specific tasks** that it may not be able to do on its own. 

They are also a way of transforming **free human text input** into **structured inputs to external tools** e.g. APIs.

We can turn our own python functions into tools using the `tool` decorator from `langchain_core.tools`, or use one of many pre-built tools from LangChain/LangGraph. 

In the below example we will use a simple DuckDuckGo search engine to allow our LLM to search the web.

In order to extend the functionality of our chat model, we need to **bind** the tool(s) we want to it:

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()
search_llm = llm.bind_tools([search_tool])

Now we have a new chat model, which has the optional capability to perform a web search. 
- If we ask it a question that it "knows" already from training, it may not use the tool(s) we gave it, and just answer. 
- If the model believes that it needs to use the tool provided to give us our answer, it will generate `Tool Call`(s), which each include the specific tool it thinks it needs, as well as any arguments needed to be passed into it, **in the structure that that tool requires:**

This behaviour is also model-dependent.

In [ ]:
search_llm.invoke("what is 2+2?").pretty_print()

In [ ]:
search_llm.invoke('who is the uk prime minister today?').pretty_print()

The above output tells us that the LLM thinks we should search the web to get the answer to our query, and what the input to the search engine would be

What we can do now is add this search functionality as a **node** to our graph using a pre-built `ToolNode` and a `tools_condition` for our conditional edge, which decides whether to use the tool or go to the end of the graph:

Adding the option to search the web to our graph:

In [ ]:
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

def tool_calling_llm(state: MessagesState):
    return {"messages": [search_llm.invoke(state["messages"])]}

# Build graph
workflow = StateGraph(MessagesState)

#this is the central node that decides whether to search the web:
workflow.add_node("LLM that knows how to web search", tool_calling_llm)

#This is the node that will actually perform the search:
workflow.add_node("tools", ToolNode([search_tool]))

# Now add the logical flow:
workflow.add_edge(START, "LLM that knows how to web search")
workflow.add_conditional_edges(
    "LLM that knows how to web search",
    # If the latest message (result) from central LLM is a tool call -> tools_condition routes to tools
    # If the latest message (result) from central LLM is a not a tool call -> tools_condition routes to END
    tools_condition,
)

#we can route the result of the tool back to the central LLM, 
# which can give us a summary answer based on the search results:
workflow.add_edge("tools", "LLM that knows how to web search") 

#or we can route the result of the tool to the end of the graph, if we don't want to summarise it:
#workflow.add_edge("tools", END)
graph = workflow.compile()

# View
display(Image(graph.get_graph(xray=True).draw_mermaid_png(curve_style=CurveStyle.BASIS)))


Now we can have an LLM at the centre of our graph which has the *option* of using a web search tool and can *decide* whether to use this tool to give us a response:

In [ ]:
import time
current_date = time.strftime("%d.%m.%Y")

#feel free to play around with this:
system_prompt_text =  f"""You are a helpful assistant who, if needed, will search the web for the latest information, use that as the basis for your response, and summarise the search results to the user. 
In your web searches, if searching for the latest information, use 'latest' or 'current' or 'today' instead of hard-coding the year. 
If the web search results refer to a future date from your knowledge cutoff, you should use them in your response. The current date is {current_date}."""


system_prompt = SystemMessage(content=system_prompt_text)


In [ ]:
france_query = graph.invoke({"messages": [system_prompt, HumanMessage(content='what is the capital of France?')]})
for message in france_query["messages"]:
    message.pretty_print()


In [ ]:
chat_history = graph.invoke({"messages": [system_prompt, HumanMessage(content="what is the name of the current UK prime minister?")]})
for message in chat_history["messages"]:
    message.pretty_print()

In the above example we have a new type of `Message` in our chat history, the `Tool Message`:
- The `Tool Call` is being formulated in the `AI Message` above it, where the llm:
    - decides that it needs a tool
    - decides that it specifically needs the tool `duckduckgo_search`
    - it converts our plain text `HumanMessage` into an argument to send to that tool, in this case, a search query for DuckDuckGO
    - it routes this query to the tool node 

- The search itself is performed in the ToolNode and the results are returned as a `ToolMessage`, the contents of which are basically the return value of the duckduckgo_search function.

The llm then: 
- **summarises** the results of the query 
- decides not to use any more tools
- sends us the summary as the output, and the Graph flow ends.



### Bias Detection Tool

We will be working with a package called [UnbiasPlus](https://vectorinstitute.github.io/unbias-plus/), which uses a [fine-tuned version](https://huggingface.co/vector-institute/Qwen3-8B-UnBias-Plus-SFT-Instruct-V2) of `Qwen3-8B`. 

We have made it deployable it as a HuggingFace endpoint for this workshop using a [customised verion](https://huggingface.co/stasiafromberms/Qwen3-8B-UnBias-Plus-SFT-Instruct-V2) of the model repository. 

NB: For this workshop, you will need to deploy [your own endpoint](https://endpoints.huggingface.co)

UnbiasPlus is an open-source [python package](https://vectorinstitute.github.io/unbias-plus/) from The Vector Institute that is trained to remove bias from journalistic texts (see paper [here](https://huggingface.co/papers/2606.23412)).

In the below, we will attach the Endpoint API to our graph as a ToolNode, and have our agent send inputs to it if needed:

In [ ]:
from langchain_core.tools import tool
import requests

@tool
def bias_detection_tool(inputs:str):
    """
    A tool that detects bias in a given text
    Args: inputs: str, the text to be analyzed for bias
    Returns: response dictionary including suggested re-write, as well as a brief explanation of why the text is considered biased or unbiased

    """
    
    # Make sure these are correctly set in your .env file or environment variables, 
    # and if needed, re-run load_dotenv() to refresh the environment variables, 
    # or refresh the jupyter kernel.

    ENDPOINT_URL = os.getenv("HF_ENDPOINT")
    HF_TOKEN = os.getenv("HF_TOKEN")  

    # Setting up headers for the request to the Hugging Face endpoint
    headers = {
    "Authorization": f"Bearer {HF_TOKEN}",
    "Content-Type": "application/json",
    }

    # The reqest payload sent to the model:
    payload = {"inputs": inputs}

    # Send request:
    #print(f"Pinging your Endpoint Handler at: {ENDPOINT_URL}...")
    try:
        response = requests.post(ENDPOINT_URL, headers=headers, json=payload)
        response.raise_for_status()
    
        return response.json()['result']

    except requests.exceptions.HTTPError as http_err:
        print(f"\n HTTP Error: {http_err}")
        print(f"Response Body: {response.text}")
    except Exception as err:
        print(f"\n General Connection Error: {err}")
    
    


Binding this tool to our LLM:

In [ ]:
llm_with_bias_tool = llm.bind_tools([bias_detection_tool])

And creating our Agent graph:

In [ ]:
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

# Our main agent node that is able to also call the bias detection tool:
def anti_bias_agent(state: MessagesState):
    return {"messages": [llm_with_bias_tool.invoke(state["messages"])]}

# Build graph
workflow = StateGraph(MessagesState)

#this is the central node that decides whether to use the bias agent:
workflow.add_node("Main Agent Node", anti_bias_agent)

#This is the node that will use the tool:
workflow.add_node("tools", ToolNode([bias_detection_tool]))

# Now add the logical flow:
workflow.add_edge(START, "Main Agent Node")
workflow.add_conditional_edges(
    "Main Agent Node",
    # If the latest message (result) from central LLM is a tool call -> tools_condition routes to tools
    # If the latest message (result) from central LLM is a not a tool call -> tools_condition routes to END
    tools_condition,
)

#we route the result of the tool back to the central LLM, 
# which can give us a summary answer based on the  tool call results, or call the tool(s) again:
workflow.add_edge("tools", "Main Agent Node") 

graph = workflow.compile()

# View
display(Image(graph.get_graph(xray=True).draw_mermaid_png(curve_style=CurveStyle.BASIS)))

Now we can play around with the system prompt a bit and see what happens:

In [ ]:
system_prompt_text =  f"""You are a helpful assistant tasked with detecting and removing bias in texts. 
If provided with an input that contains bias, please identify the sections that are problematic, and explain why they are biased, and suggest a more neutral version of the text. 
If the input text is already neutral, please confirm that it is unbiased and provide a brief explanation of why it is considered unbiased.
"""

system_prompt = SystemMessage(content=system_prompt_text)

In [ ]:
human_input = HumanMessage(content="Hi, is this an ok job ad? 'looking for young and attractive office manager to join our team'")

In [ ]:
chat_history = graph.invoke({"messages": [system_prompt, human_input]})
for message in chat_history["messages"]:
    message.pretty_print()

Now we can try it with a longer text:

In [ ]:
JOB_AD = """
what we're looking for: someone who's extremely online and just gets internet culture, no explaining memes to you
warm, bubbly energy that makes people feel instantly comfortable
comfortable being visible, an existing personal brand or decent following is a big plus
happy to wear a lot of hats, this isn't a "not my job" kind of place
available and responsive outside typical hours, we move at startup speed
"""

In [ ]:
human_input = HumanMessage(content=f"Hi, is this an ok job ad? '{JOB_AD}'")

In [ ]:
chat_history = graph.invoke({"messages": [system_prompt, human_input]})
for message in chat_history["messages"]:
    message.pretty_print()

## Your Turn!

Now it is your turn! Feel free to play around with:
- the agent system prompt
- the architecture
- the tool(s) the LLM has access to
- the underlying model
- ... anything else!

There are some `job_postings` in this repo, you can try it out with them, or with your own text snippets